In [1]:
!pip install langchain
!pip install langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 4.3 MB/s eta 0:00:00


In [2]:
from getpass import  getpass
import os
OPENAI_KEY = getpass('Enter Open AI API Key: ')
os.environ['OPENAI_API_KEY'] = OPENAI_KEY


Enter Open AI API Key: ··········


In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model_name='gpt-4o', temperature=0)

In [4]:
it_support_queue = [
    "I can't access my email. It keeps showing an error message. Please help.",
    "Tengo problemas con la VPN. No puedo conectarme a la red de la empresa. ¿Pueden ayudarme, por favor?",
    "Mon imprimante ne répond pas et n'imprime plus. J'ai besoin d'aide pour la réparer.",
    "我无法访问公司的网站。每次都显示错误信息。请帮忙解决。"
]

it_support_queue

["I can't access my email. It keeps showing an error message. Please help.",
 'Tengo problemas con la VPN. No puedo conectarme a la red de la empresa. ¿Pueden ayudarme, por favor?',
 "Mon imprimante ne répond pas et n'imprime plus. J'ai besoin d'aide pour la réparer.",
 '我无法访问公司的网站。每次都显示错误信息。请帮忙解决。']

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt1 = """
  Act as a customer support agent.
  For the customer support message delimited below by triple backticks,
  Output the language of the message in one word only, e.g. Spanish

  Customer Message:
  ```{orig_msg}```
"""

prompt_template1 = ChatPromptTemplate.from_template(prompt1)

chain_1 = (prompt_template1 | llm | StrOutputParser())

In [8]:
chain_1.invoke({'orig_msg':"I can't access my email. It keeps showing an error message. Please help." })

'English'

In [11]:
# Okay to stroe the response returned from chain_1, we can use a Runnable Passthrough

from langchain_core.runnables import RunnablePassthrough

RunnablePassthrough.assign(orig_lang = chain_1).invoke({'orig_msg': it_support_queue[1]})

{'orig_msg': 'Tengo problemas con la VPN. No puedo conectarme a la red de la empresa. ¿Pueden ayudarme, por favor?',
 'orig_lang': 'Spanish'}

In [12]:
# Chain 2: Translate Customer Message to English
prompt2 = """
  Act as a customer support agent.
  For the customer message and customer message language delimited below by triple backticks,
  Translate the customer message from the customer message language to English
  if customer message language is not in English,
  else return back the original customer message.

  Customer Message:
  ```{orig_msg}```
  Customer Message Language:
  ```{orig_lang}```
"""

prompt_template2 = ChatPromptTemplate.from_template(prompt2)

chain_2 = (prompt_template2 | llm | StrOutputParser())

In [14]:
resp =RunnablePassthrough.assign(trans_msg = chain_2).invoke({'orig_msg': it_support_queue[1], 'orig_lang': chain_1.invoke({'orig_msg':"I can't access my email. It keeps showing an error message. Please help." })})

type(resp)

dict

In [21]:
# Chain 3: Generate a resolution response in English
prompt3 = """
  Act as a customer support agent.
  For the customer support message delimited below by triple backticks,
  Generate an appropriate resolution response in English.

  Customer Message:
  ```{trans_msg}```
"""
prompt_template3 = ChatPromptTemplate.from_template(prompt3)

chain_3 = (prompt_template3 | llm | StrOutputParser())


In [16]:
# Chain 4: Translate resolution response from English to Customer's original language
prompt4 = """
  Act as a customer support agent.
  For the customer resolution response and target language delimited below by triple backticks,
  Translate the customer resolution response message from English to the target language
  if target language is not in English,
  else return back the original customer resolution response.

  Customer Resolution Response:
  ```{trans_response}```
  Target Language:
  ```{orig_lang}```
"""
prompt_template4 = ChatPromptTemplate.from_template(prompt4)

chain_4 = (prompt_template4 | llm | StrOutputParser())

In [22]:
final_chain = (
    RunnablePassthrough.assign(orig_lang = chain_1)
                  |
    RunnablePassthrough.assign(trans_msg = chain_2)
                  |
    RunnablePassthrough.assign(trans_response = chain_3)
                  |
    RunnablePassthrough.assign(final_response = chain_4)

    )


In [23]:
formatted_msgs = [{'orig_msg' : msg} for msg in it_support_queue ]

In [24]:
response = final_chain.map().invoke(formatted_msgs)

In [26]:
import pandas as pd

pd.DataFrame(response)

,orig_msg,orig_lang,trans_msg,trans_response,final_response
0,I can't access my email. It keeps showing an e...,English,I can't access my email. It keeps showing an e...,I'm sorry to hear that you're having trouble a...,I'm sorry to hear that you're having trouble a...
1,Tengo problemas con la VPN. No puedo conectarm...,Spanish,I am having trouble with the VPN. I cannot con...,Thank you for reaching out to us. I understand...,Gracias por contactarnos. Entiendo lo importan...
2,Mon imprimante ne répond pas et n'imprime plus...,French,My printer is not responding and is no longer ...,I'm sorry to hear that your printer is not res...,Je suis désolé d'apprendre que votre imprimant...
3,我无法访问公司的网站。每次都显示错误信息。请帮忙解决。,Chinese,I am unable to access the company's website. I...,I'm sorry to hear that you're experiencing dif...,很抱歉听到您在访问我们的网站时遇到了困难。让我们一起努力解决这个问题。您可以尝试以下几个步骤...


In [27]:
while True:
  print('ok')

Streaming output truncated to the last 5000 lines.
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
o

KeyboardInterrupt: 